# G11 - the rank-wall account, predicted then tested

Seven accounts of the cliff have been tested and all seven falsified - six on shape (smooth where the outcome is discontinuous), and the head-target account on location. G9 produced an eighth candidate that survived: the cliff sits at the SOURCE ambient dimension (img_small 768-d, cliff at 768) and does not move with the target.

**The prediction prints before any measurement:** img_base cliffs at 1536, img_large at 2048, neither moving with the target. G9 could not test these - its grid stopped at 1024, so both are genuinely out of sample.

A hit makes this the eighth account tested and the first to survive. A miss makes it the eighth falsified.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G11 — the rank-wall account, stated as a prediction and then tested.
# Run AFTER G9. Requires G0-exact.
#
# WHERE THIS COMES FROM. SEVEN accounts of the width cliff have now been
# tested and all seven falsified:
#
#   1  local-structure loss                        C.13.11
#   2  spectral amplification (the project's own)  C.13.11
#   3  accumulated noise energy                    G8, D = 0.55
#   4  noise/signal direction ratio                G8, D = 0.39
#   5  condition number                            G8, D = 0.67
#   6  shared-direction fraction                   G8, D = 0.88
#   7  head-target dimension                       G9 - the location does
#      not move when the target changes from bge (1024-d) to SBERT (768-d)
#
# The first six failed for the same reason: each is SMOOTH where the
# outcome is discontinuous. The seventh failed on a different and cleaner
# ground - it predicted a location that was not observed.
#
# G9 also produced an EIGHTH candidate, and this one survived:
#
#   img_small (768-d source), target bge  (1024-d): cliff at 768, ratio 28.9
#   img_small (768-d source), target sbert (768-d): cliff at 768, ratio 18.8
#
# Same location under both targets, at exactly the SOURCE encoder's
# ambient dimension. Note also that the two RATIOS differ (28.9 vs 18.8)
# while the location does not - a quantity that fixes WHERE the break
# happens and leaves HOW BAD to something else is what a structural
# threshold looks like.
#
# THE MECHANISM. A d-dimensional source's ridge entry map produces hub
# coordinates spanning at most d directions. The head is fitted on those
# coordinates, so it has never seen the remaining hub directions. Past
# width d, the other encoders populate directions the head cannot read,
# and transfer does not degrade - it collapses. 0.952 to 0.032 with bge,
# 0.963 to 0.038 with SBERT: roughly three per cent of native, which is
# annihilation rather than degradation. Conditioning problems degrade;
# rank walls do this. That is a threshold BY CONSTRUCTION, not a curve,
# and it is the first candidate whose SHAPE matches the phenomenon.
#
# WHY THIS NOTEBOOK EXISTS. Fitting one source is not an explanation. G9
# could only test img_small, because img_base (1536-d) and img_large
# (2048-d) have dimensions beyond its 1024 grid - their absence of a cliff
# there is uninformative, not counterevidence.
#
# THE PREDICTION, WRITTEN DOWN BEFORE RUNNING:
#
#   img_base  (1536-d) cliffs between 1408 and 1536, and nowhere earlier.
#   img_large (2048-d) cliffs between 1920 and 2048, and nowhere earlier.
#   Neither location moves when the head target changes.
#
# WHAT FALSIFIES IT. Any of: a cliff at a width unrelated to the source's
# dimension; no cliff at all where one is predicted; or a location that
# shifts with the target. Any of those and the rank wall joins the ledger
# as the EIGHTH falsified account of the cliff, and the cliff returns to
# unexplained with one more family excluded.
#
# COST. The concat is 5,376-d and 8,533 rows, so widths up to 2,048 are
# within rank. One SVD, sliced. The grid is coarse away from the predicted
# points and fine near them - there is no reason to spend resolution where
# nothing is expected.
# ==========================================================
import os
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ["DATA_DIR"])
ALPHA, N_EVAL, SEED = 1e-2, 1000, 0
CLIFF_RATIO = 3.0
TOL = 128           # a cliff counts as "at d" if within one grid step

# coarse where nothing is predicted, fine at 768 / 1536 / 2048
WIDTHS = [512, 640, 768, 896, 1152, 1408, 1536, 1664, 1920, 2048, 2176]

SPACES = {}
for size in ("small", "base", "large"):
    SPACES[f"img_{size}"] = np.load(
        str(DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz")
    )["img"].astype(np.float64)
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
SPACES["txt_bge"] = np.load(
    str(DATA_DIR / "crossmodal_pairs.npz"))["txt"].astype(np.float64)[:N]

TARGETS = {"bge": SPACES["txt_bge"]}
_sb = DATA_DIR / "e13_txt_sbert.npz"
if _sb.exists():
    TARGETS["sbert"] = np.load(str(_sb))["txt"].astype(np.float64)[:N]

SOURCES = ["img_small", "img_base", "img_large"]
PREDICTED = {s: SPACES[s].shape[1] for s in SOURCES}
print("PREDICTION, fixed before running:")
for s in SOURCES:
    print(f"  {s:12s} ({PREDICTED[s]:5d}-d) cliffs at {PREDICTED[s]}, "
          f"and nowhere earlier")
print("  and no location moves with the head target\n")

rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
te, tr = perm[:N_EVAL], perm[N_EVAL:]


def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)


def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)


def r1(P, G):
    return float(((l2n(P) @ l2n(G).T).argmax(1) == np.arange(len(P))).mean())


_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_u, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)
usable = [w for w in WIDTHS if w <= len(_sv)]
print(f"concat {_ref.shape}; widths swept: {usable}")
if len(usable) < len(WIDTHS):
    print(f"  dropped {[w for w in WIDTHS if w > len(_sv)]} - beyond concat rank")


def hub_at(d):
    B = _VT[:d].T / (_sv[:d] / np.sqrt(len(_ref)))
    return (_ref - _mu) @ B


def sweep(src, tgt_name):
    T = TARGETS[tgt_name]
    gal = l2n(T[te])
    out = []
    for d in usable:
        H = hub_at(d)
        to_hub = {k: ridge(SPACES[k][tr], H) for k in SOURCES}
        head = ridge(SPACES[src][tr] @ to_hub[src], T[tr])
        pcts = []
        for enc in SOURCES:
            if enc == src:
                continue
            zero = r1((SPACES[enc][te] @ to_hub[enc]) @ head, gal)
            nat = r1(SPACES[enc][te] @ ridge(SPACES[enc][tr], T[tr]), gal)
            pcts.append(zero / max(nat, 1e-9))
        out.append(float(np.mean(pcts)))
    return np.array(out)


MIN_DROP = 0.02      # a step smaller than this is not a cliff at any ratio


def find_cliff(curve):
    """Largest negative step, guarded two ways.

    The ratio alone is not enough: on a nearly flat curve the denominator
    goes to zero and any dip scores in the trillions. So the drop must
    also be materially large in absolute terms, and the denominator is
    floored - otherwise a 0.001 wobble on a flat line reads as a cliff.
    """
    steps = np.diff(curve)
    i = int(np.argmin(steps))
    if steps[i] >= -MIN_DROP:
        return None, 0.0
    others = np.abs(np.delete(steps, i))
    denom = max(others.max(), MIN_DROP / CLIFF_RATIO)
    ratio = min(abs(steps[i]) / denom, 999.0)
    return (usable[i + 1] if ratio >= CLIFF_RATIO else None), ratio


print("\n" + "=" * 100)
print(f"{'source':12s}{'dim':>6s}{'target':>8s}  "
      + "".join(f"{w:>7d}" for w in usable) + f"{'cliff':>8s}{'ratio':>7s}")
print("=" * 100)
results = {}
for tgt in TARGETS:
    for src in SOURCES:
        c = sweep(src, tgt)
        at, ratio = find_cliff(c)
        results[(src, tgt)] = (at, ratio)
        print(f"{src:12s}{PREDICTED[src]:>6d}{tgt:>8s}  "
              + "".join(f"{v:>7.3f}" for v in c)
              + (f"{at:>8d}" if at else f"{'none':>8s}") + f"{ratio:>7.1f}")

In [ ]:
# ---------- score the prediction ----------
print("\n" + "=" * 100)
hits, misses, moved = [], [], []
for src in SOURCES:
    locs = {results[(src, t)][0] for t in TARGETS}
    d = PREDICTED[src]
    ok = locs and all(a is not None and abs(a - d) <= TOL for a in locs)
    (hits if ok else misses).append(src)
    if len(locs) > 1:
        moved.append(src)
    print(f"  {src:12s} predicted {d:5d}   observed {locs or '{none}'}"
          + ("   HIT" if ok else "   MISS"))

print()
if len(hits) == len(SOURCES) and not moved:
    print("PREDICTION CONFIRMED ON ALL THREE SOURCES.")
    print("The width cliff is a RANK WALL: it sits at the source encoder's")
    print("ambient dimension, moves when that dimension moves, and ignores")
    print("the head target. Past that width the head is asked to read hub")
    print("directions its source could never populate.")
    print()
    print("This is the EIGHTH account of the cliff tested and the first to")
    print("survive - and it survived a prediction written down before the")
    print("measurement, not a fit to data already seen. The cliff is")
    print("explained.")
    print()
    print("Two things it does NOT license. It is one hub protocol and one")
    print("image corpus. And it does not retroactively rescue the seven")
    print("falsified accounts - they were tested against the same curve and")
    print("failed on shape; this one succeeds for a reason none of them had.")
elif hits and misses:
    print(f"PARTIAL: {', '.join(hits)} hit, {', '.join(misses)} missed.")
    print("A rank wall that holds for some sources and not others is not a")
    print("rank wall. Report the table and treat the account as unresolved -")
    print("do not keep the sources that fit and drop the ones that do not.")
else:
    print("PREDICTION FALSIFIED. The cliff does not track the source")
    print("dimension outside the case that suggested it, so the rank-wall")
    print("account becomes the EIGHTH falsified account of the cliff. G9's")
    print("img_small result stands as an observation still in need of an")
    print("explanation.")
if moved:
    print(f"\n  Location moved with the target for: {', '.join(moved)}.")
    print("  That contradicts the account independently of the hit/miss")
    print("  count, since a rank wall cannot depend on what is being read.")